# Random Forest

In [1]:
# 1) imports básicos
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 2) importar tudo do notebook de pré-processamento
#    -> isto vai definir: full_train_dataset, cat_feat, num_feat,
#       basic_string_transformer, preprocess_categorical, funções fit_/transform_,
#       MyTargetEncoder, MyOneHotEncoder, etc.
%run 05_0_preproc_helpers.ipynb  # ajusta o caminho se estiver noutra pasta

# 3) definir a coluna target (ajusta se o nome for outro)
TARGET_COL = "price"  # <-- MUITO IMPORTANTE: confirma o nome no teu CSV

# 4) separar X e y a partir do dataset já tratado no helper
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# usar as listas já definidas no helper
numeric_features = num_feat                      # ['year', 'mileage', ...]
categorical_features = cat_feat                  # ['Brand', 'model', 'transmission', 'fuelType']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)


X shape: (75973, 11)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'paintQuality%', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [2]:
# -----------------------------
# DEFINIÇÕES GERAIS
# -----------------------------
N_SPLITS = 5
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# -----------------------------
# ESPAÇO DE HIPERPARÂMETROS
# -----------------------------
param_distributions = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "max_depth": [5, 7, 9, 11, 15, 20],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None],
    "bootstrap": [True, False]
}

N_RANDOM_CONFIGS = 40  # ajusta consoante o tempo de treino

sampler = ParameterSampler(
    param_distributions,
    n_iter=N_RANDOM_CONFIGS,
    random_state=RANDOM_STATE
)

search_results = []
best_rmse = np.inf
best_config = None

log_path = "rf_random_search_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg)
        log_file.write(msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# INÍCIO DO RANDOM SEARCH RF")
    log("# =============================")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")

    # -----------------------------
    # RANDOM SEARCH + CV
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parâmetros: {params}")

        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        # CV para esta configuração
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] ==== FOLD {fold}/{N_SPLITS} ====")

            # ---------------------------------
            # 1) separar train / validation
            # ---------------------------------
            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # NaNs ANTES da imputação numérica
            log(f"[C{config_id}|F{fold}] NaNs ANTES imputação numérica (apenas numéricas):")
            log(str(X_train[numeric_features].isna().sum()))

            log(f"[C{config_id}|F{fold}] NaNs ANTES (categóricas):")
            log(str(X_train[categorical_features].isna().sum()))

            unknown_counts = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' ANTES (categóricas):")
            log(str(unknown_counts))


            # ---------------------------------
            # 2) PREPROCESSING NUMÉRICO (fit no train, transform no val)
            # ---------------------------------

            # 2.1) year com median por modelo
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2) mileage
            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3) engineSize
            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # 2.4) tax
            tax_state = fit_tax_imputer(
                X_train,
                tax_col="tax",
                do_abs=True
            )
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            # 2.5) mpg
            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            # 2.6) paintQuality%
            paint_state = fit_paint_quality_imputer(
                X_train,
                paint_col="paintQuality%"
            )
            X_train = transform_paint_quality_imputer(X_train, state=paint_state)
            X_val   = transform_paint_quality_imputer(X_val,   state=paint_state)

            # 2.7) previousOwners
            owners_state = fit_previous_owners_imputer(
                X_train,
                owners_col="previousOwners",
                year_col="year",
                mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            log(f"[C{config_id}|F{fold}] Após imputação numérica: X_train shape = {X_train.shape}, X_val shape = {X_val.shape}")
            log(f"[C{config_id}|F{fold}] NaNs DEPOIS imputação numérica (apenas numéricas):")
            log(str(X_train[numeric_features].isna().sum()))

            # ---------------------------------
            # 3) ENCODING CATEGÓRICO (Target p/ Brand+model, One-Hot p/ resto)
            # ---------------------------------
            high_card_features = ["Brand", "model"]  # target encoding
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            log(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")

            # 3.1) Target Encoding para Brand e model
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)

            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-Hot Encoding para as restantes categóricas
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])

            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")

            log(f"[C{config_id}|F{fold}] NaNs DEPOIS encoding (X_train_cat, total): {X_train_cat.isna().sum().sum()}")
            log(f"[C{config_id}|F{fold}] NaNs DEPOIS encoding (X_val_cat, total): {X_val_cat.isna().sum().sum()}")
            
            
            unknown_cols_train = [col for col in X_train_cat.columns if "UNKNOWN" in str(col)]
            unknown_cols_val   = [col for col in X_val_cat.columns if "UNKNOWN" in str(col)]

            log(f"[C{config_id}|F{fold}] Colunas encodadas com 'UNKNOWN' (train): {unknown_cols_train}")
            log(f"[C{config_id}|F{fold}] Colunas encodadas com 'UNKNOWN' (val)  : {unknown_cols_val}")

            # ---------------------------------
            # 4) SCALING NUMÉRICO
            # ---------------------------------
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features])
            X_val_num   = scaler.transform(X_val[numeric_features])

            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features]
            )

            log(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # ---------------------------------
            # 5) JUNTAR NUMÉRICO + CATEGÓRICO
            # ---------------------------------
            X_train_final = pd.concat(
                [X_train_num_df, X_train_cat],
                axis=1
            )
            X_val_final = pd.concat(
                [X_val_num_df, X_val_cat],
                axis=1
            )

            log(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # ---------------------------------
            # 6) MODELO: RANDOM FOREST COM OS PARÂMETROS DA CONFIG ATUAL
            # ---------------------------------
            rf = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **params
            )

            log(f"[C{config_id}|F{fold}] A treinar RandomForestRegressor...")
            rf.fit(X_train_final, y_train)
            y_pred = rf.predict(X_val_final)

            # ---------------------------------
            # 7) MÉTRICAS POR FOLD
            # ---------------------------------
            mse  = mean_squared_error(y_val, y_pred)
            rmse = np.sqrt(mse)
            mae  = mean_absolute_error(y_val, y_pred)
            r2   = r2_score(y_val, y_pred)

            log(f"[C{config_id}|F{fold}] RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

            fold_rmses.append(rmse)
            fold_maes.append(mae)
            fold_r2s.append(r2)

        # MÉDIA DOS FOLDS PARA ESTA CONFIG
        mean_rmse = np.mean(fold_rmses)
        mean_mae  = np.mean(fold_maes)
        mean_r2   = np.mean(fold_r2s)

        log("")
        log(f"Config {config_id} - RMSE médio: {mean_rmse:.4f} | MAE médio: {mean_mae:.4f} | R² médio: {mean_r2:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_mean": mean_rmse,
            "mae_mean": mean_mae,
            "r2_mean": mean_r2
        })

        # guardar melhor até agora (minimizar RMSE)
        if mean_rmse < best_rmse:
            best_rmse = mean_rmse
            best_config = {**params}
            log(f"[NOVO MELHOR] Config {config_id} com RMSE médio = {best_rmse:.4f}")

    log("")
    log("# =============================")
    log("# FIM DO RANDOM SEARCH RF")
    log("# =============================")
    log(f"Melhor configuração encontrada: {best_config}")
    log(f"Melhor RMSE médio: {best_rmse:.4f}")

# -----------------------------
# RESUMO FINAL DO RANDOM SEARCH EM DATAFRAME
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True)

display(results_df_sorted.head(10))

print("\nMelhor configuração encontrada (min RMSE):")
print(best_config)
print("Melhor RMSE médio:", best_rmse)
print(f"\nLogs detalhados guardados em: {log_path}")


# =============================
# INÍCIO DO RANDOM SEARCH RF
# =============================
N_SPLITS = 5, N_RANDOM_CONFIGS = 40
param_distributions = {'n_estimators': [200, 400, 600, 800, 1000], 'max_depth': [5, 7, 9, 11, 15, 20], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 4, 8], 'max_features': [None], 'bootstrap': [True, False]}

######## CONFIG 1/40 ########
Parâmetros: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 15, 'bootstrap': False}

[CONFIG 1] ==== FOLD 1/5 ====
[C1|F1] X_train shape: (60778, 11), X_val shape: (15195, 11)
[C1|F1] y_train shape: (60778,), y_val shape: (15195,)
[C1|F1] NaNs ANTES imputação numérica (apenas numéricas):
year              1200
mileage           1158
engineSize        1227
tax               6361
mpg               6360
paintQuality%     1217
previousOwners    1206
dtype: int64
[C1|F1] NaNs ANTES (categóricas):
Brand           0
model           0
transmission    0
fuelTyp

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_mean,mae_mean,r2_mean
7,8,1000,6,1,None,15,True,2323.673179,1376.896327,0.942920
23,24,600,2,2,None,15,True,2342.836876,1376.135907,0.941965
2,3,200,6,2,None,15,True,2353.418683,1378.917887,0.941456
17,18,1000,2,8,None,20,True,2498.985098,1421.996900,0.934042
1,2,600,10,8,None,20,True,2499.182427,1422.142380,0.934031
5,6,400,2,2,None,11,True,2525.348114,1545.970343,0.932636
36,37,400,10,8,None,11,True,2635.016449,1566.408154,0.926697
38,39,800,4,1,None,9,True,2758.810763,1734.319476,0.919676
33,34,200,4,1,None,9,True,2759.730525,1734.821860,0.919626
0,1,400,10,2,None,15,False,2780.696044,1606.246049,0.918322



Melhor configuração encontrada (min RMSE):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Melhor RMSE médio: 2323.673178906674

Logs detalhados guardados em: rf_random_search_log.txt


In [3]:
# -----------------------------
# 1) SEPARAR FEATURES DE TREINO (CÓPIA)
# -----------------------------
X_full = X.copy()
y_full = y.copy()

# se ainda não o tiveres explícito:
high_card_features = ["Brand", "model"]
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# -----------------------------
# 2) PREPROCESSING NUMÉRICO NO TREINO (fit em X_full)
# -----------------------------

# 2.1) year com median por modelo
year_state = fit_year_median(
    X_full,
    year_col="year",
    model_col="model"
)
X_full = transform_year_with_model_median(X_full, state=year_state)

# 2.2) mileage
mileage_state = fit_mileage_imputer(
    X_full,
    mileage_col="mileage",
    do_abs=True
)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# 2.3) engineSize
engine_state = fit_engine_size_imputer(
    X_full,
    engine_col="engineSize"
)
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# 2.4) tax
tax_state = fit_tax_imputer(
    X_full,
    tax_col="tax",
    do_abs=True
)
X_full = transform_tax_imputer(X_full, state=tax_state)

# 2.5) mpg
mpg_state = fit_mpg_imputer(
    X_full,
    mpg_col="mpg",
    do_abs=True
)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# 2.6) paintQuality%
paint_state = fit_paint_quality_imputer(
    X_full,
    paint_col="paintQuality%"
)
X_full = transform_paint_quality_imputer(X_full, state=paint_state)

# 2.7) previousOwners
owners_state = fit_previous_owners_imputer(
    X_full,
    owners_col="previousOwners",
    year_col="year",
    mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

# -----------------------------
# 3) ENCODING CATEGÓRICO NO TREINO
#    Target p/ Brand+model, One-Hot p/ resto
# -----------------------------

# 3.1) Target Encoding p/ Brand e model
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)

X_full_high = te.transform(X_full[high_card_features])

# 3.2) One-Hot p/ transmission, fuelType
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])

X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

# -----------------------------
# 4) SCALING NUMÉRICO NO TREINO
# -----------------------------
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# -----------------------------
# 5) MATRIZ FINAL DE TREINO
# -----------------------------
X_full_final = pd.concat(
    [X_full_num_df, X_full_cat],
    axis=1
)

print("X_full_final shape:", X_full_final.shape)

# -----------------------------
# 6) TREINAR RANDOM FOREST FINAL
# -----------------------------
rf_best = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **best_config  # vem do random search
)

rf_best.fit(X_full_final, y_full)

X_full_final shape: (75973, 43)


RandomForestRegressor(max_depth=15, max_features=None, min_samples_split=6,
                      n_estimators=1000, n_jobs=-1, random_state=42)

In [5]:
# -----------------------------
# 1) PREPARAR FEATURES DO TESTE
# -----------------------------
# garante que o test_df tem as mesmas colunas de features
test_df = pd.read_csv("../../project_data/test.csv")  
X_test = test_df[numeric_features + categorical_features].copy()

# -----------------------------
# 2) MESMO PREPROCESSING NUMÉRICO (só transform)
# -----------------------------

X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# -----------------------------
# 3) MESMO ENCODING CATEGÓRICO
# -----------------------------
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])

X_test_cat = pd.concat([X_test_high, X_test_low], axis=1)

# -----------------------------
# 4) MESMO SCALING NUMÉRICO
# -----------------------------
X_test_num = scaler.transform(X_test[numeric_features])

X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# -----------------------------
# 5) MATRIZ FINAL DE TESTE
# -----------------------------
X_test_final = pd.concat(
    [X_test_num_df, X_test_cat],
    axis=1
)

print("X_test_final shape:", X_test_final.shape)

# -----------------------------
# 6) PREVISÕES NO TEST SET
# -----------------------------
y_test_pred = rf_best.predict(X_test_final)

# quick sanity-check
print("Primeiras 10 previsões:", y_test_pred[:10])
print("Resumo das previsões:")
print(pd.Series(y_test_pred).describe())


X_test_final shape: (32567, 43)
Primeiras 10 previsões: [25579.03604072 21772.82742576 12353.05133962 21118.37949562
 23642.15194336 13669.73492021 15000.00390686 15224.96279112
  6031.0746955  24612.5406445 ]
Resumo das previsões:
count    32567.000000
mean     17563.409290
std       7474.664934
min       2128.352029
25%      13409.123872
50%      15866.576921
75%      20055.099680
max      69850.473433
dtype: float64


In [8]:
y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred_round
})

submission.head()


,carID,price
0,89856,25579
1,106581,21773
2,80886,12353
3,100174,21118
4,81376,23642


In [9]:
submission_path = "rf_best_submission.csv"
submission.to_csv(submission_path, index=False)
print("Ficheiro de submissão criado em:", submission_path)


Ficheiro de submissão criado em: rf_best_submission.csv
